In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
max_seq_length = 4096 
load_in_4bit = True 
path_model_v1 = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_model_v1, 
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
gold_data_file = "/workspaces/Arch_PC_Assistent/dataset/rsft_gold_dataset.jsonl"
dataset = load_dataset("json", data_files=gold_data_file, split="train")

def formatting_prompts_func(examples):
    texts = []
    # Zip allows us to iterate over instruction, input (context), and output simultaneously
    for instruction, input_ctx, output in zip(examples["instruction"], examples["input"], examples["output"]):
        
        # We must use EXACTLY the same system prompt structure used during generation
        system_prompt = f"""You are ArchAgent, an expert Arch Linux with hyprland and zsh setup assistant. 
        Below is retrieved context from the Arch Wiki. Use this context as your primary source of truth to ensure accuracy.
        
        CONTEXT:
        {input_ctx}
        
        CRITICAL INSTRUCTIONS:
        If the context is incomplete or does not fully cover the user's problem, you MUST seamlessly use your own internal knowledge to provide a complete solution.
        
        Your output MUST strictly follow this exact format:
        <think>
        [Analyze the user's problem. Check what information is available in the context. If something is missing, retrieve it from your internal knowledge. Plan your solution.]
        </think>
        <answer>
        [Your final, clear, and actionable answer here.]
        </answer>"""

        # Build the conversation exactly as the model expects it
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        
    return { "text" : texts }

# Map the formatting function to the dataset
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset loaded and formatted. Total samples: {len(formatted_dataset)}")

Map:   0%|          | 0/606 [00:00<?, ? examples/s]

Dataset loaded and formatted. Total samples: 606


In [4]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 2, # 2 epochs are plenty for a high-quality 600-sample dataset
        learning_rate = 2e-5, # Low learning rate for delicate alignment (fine-tuning)
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs/RSFT",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=17):   0%|          | 0/606 [00:00<?, ? examples/s]

In [5]:
print("Starting RSFT Training...")
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting RSFT Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 606 | Num Epochs = 2 | Total steps = 152
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.259279
2,1.310871
3,1.266218
4,1.317429
5,1.207697
6,1.163721
7,1.240780
8,1.154818
9,1.125592
10,1.138786


Unsloth: Restored added_tokens_decoder metadata in outputs/RSFT/checkpoint-152/tokenizer_config.json.


In [6]:
final_model_path = "/workspaces/Arch_PC_Assistent/outputs/RSFT/arch_assistant_final_lora"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Training complete! Final model saved to: {final_model_path}")

Unsloth: Restored added_tokens_decoder metadata in /workspaces/Arch_PC_Assistent/outputs/RSFT/arch_assistant_final_lora/tokenizer_config.json.


Training complete! Final model saved to: /workspaces/Arch_PC_Assistent/outputs/RSFT/arch_assistant_final_lora
